In [1]:
import pandas as pd

df = pd.read_csv("credit_risk_dataset.csv")

print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

Shape: (32581, 12)

Column names:
['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_status', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length']

First 5 rows:


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [2]:
# Missing values
print("Missing values:")
print(df.isnull().sum())

# Class balance
print("\nLoan Status distribution:")
print(df['loan_status'].value_counts())
print("\nPercentage:")
print(df['loan_status'].value_counts(normalize=True) * 100)

Missing values:
person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

Loan Status distribution:
loan_status
0    25473
1     7108
Name: count, dtype: int64

Percentage:
loan_status
0    78.183604
1    21.816396
Name: proportion, dtype: float64


FIXING MISSING VALUES AND OUTLIERS

In [3]:
# Fill missing values with median (robust to outliers)
df['person_emp_length'] = df['person_emp_length'].fillna(df['person_emp_length'].median())
df['loan_int_rate'] = df['loan_int_rate'].fillna(df['loan_int_rate'].median())

# Check for unrealistic ages (some datasets have age 144!)
print("Max age:", df['person_age'].max())
print("Max emp_length:", df['person_emp_length'].max())

# Remove clearly wrong rows
df = df[df['person_age'] < 100]
df = df[df['person_emp_length'] < 60]

print("\nShape after cleaning:", df.shape)
print("Missing values remaining:", df.isnull().sum().sum())

Max age: 144
Max emp_length: 123.0

Shape after cleaning: (32574, 12)
Missing values remaining: 0


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate target
X = df.drop(columns=['loan_status'])
y = df['loan_status']

# One-hot encode categorical columns
cat_cols = X.select_dtypes(include='object').columns.tolist()
print("Categorical columns:", cat_cols)

X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
print("Shape after encoding:", X.shape)
print("Columns:", X.columns.tolist())

# Split BEFORE SMOTE — test set must stay real!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTrain size:", X_train.shape)
print("Test size:", X_test.shape)
print("Default % in train:", y_train.mean() * 100)

Categorical columns: ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']
Shape after encoding: (32574, 22)
Columns: ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'person_home_ownership_OTHER', 'person_home_ownership_OWN', 'person_home_ownership_RENT', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE', 'loan_grade_B', 'loan_grade_C', 'loan_grade_D', 'loan_grade_E', 'loan_grade_F', 'loan_grade_G', 'cb_person_default_on_file_Y']

Train size: (26059, 22)
Test size: (6515, 22)
Default % in train: 21.81971679650025


In [7]:
from imblearn.over_sampling import SMOTE

print("Before SMOTE:")
print("Safe (0):", (y_train == 0).sum())
print("Default (1):", (y_train == 1).sum())

# Apply SMOTE — only on training data!
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print("Safe (0):", (y_train_sm == 0).sum())
print("Default (1):", (y_train_sm == 1).sum())
print("Total train samples:", len(X_train_sm))

Before SMOTE:
Safe (0): 20373
Default (1): 5686

After SMOTE:
Safe (0): 20373
Default (1): 20373
Total train samples: 40746


In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)  # fit on SMOTE data
X_test_scaled = scaler.transform(X_test)           # only transform test!

print("Train scaled shape:", X_train_scaled.shape)
print("Test scaled shape:", X_test_scaled.shape)

Train scaled shape: (40746, 22)
Test scaled shape: (6515, 22)


In [11]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(22,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.summary()

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_scaled, y_train_sm,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)
# Lower threshold = catch more defaulters
y_pred_low = (y_pred_prob > 0.3).astype(int)

print("With threshold 0.3:")
print(classification_report(y_test, y_pred_low, target_names=['Safe', 'Default']))

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 64)             │         1,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,097 (16.00 KB)

 Trainable params: 4,097 (16.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
1146/1146 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8622 - loss: 0.3381 - val_accuracy: 0.9298 - val_loss: 0.1579
Epoch 2/50
1146/1146 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8968 - loss: 0.2678 - val_accuracy: 0.9450 - val_loss: 0.1237
Epoch 3/50
1146/1146 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9042 - loss: 0.2485 - val_accuracy: 0.9539 - val_loss: 0.0982
Epoch 4/50
1146/1146 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9110 - loss: 0.2365 - val_accuracy: 0.9494 - val_loss: 0.1060
Epoch 5/50
1146/1146 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9146 - loss: 0.2292 - val_accuracy: 0.9519 - val_loss: 0.1009
Epoch 6/50
1146/1146 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9170 - loss: 0.2220 - val_accuracy: 0.9507 - val_loss: 0.1042
Epoch 7/50
1146/1146 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9185 - loss: 0.2183 - val_accuracy: 0.9556 - val_loss: 0.0869
Epoch 8/50
1146/1146 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9199 - loss: 0.2157 - 

In [12]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int)

loss, acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test Accuracy: {acc:.4f}\n")

print(classification_report(y_test, y_pred, target_names=['Safe', 'Default']))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

204/204 ━━━━━━━━━━━━━━━━━━━━ 0s 918us/step
Test Accuracy: 0.9228

              precision    recall  f1-score   support

        Safe       0.92      0.99      0.95      5094
     Default       0.94      0.69      0.80      1421

    accuracy                           0.92      6515
   macro avg       0.93      0.84      0.87      6515
weighted avg       0.92      0.92      0.92      6515

Confusion Matrix:
[[5029   65]
 [ 438  983]]


In [15]:
import shap
import numpy as np

# Use a small sample for speed
explainer = shap.DeepExplainer(model, X_train_scaled[:200])
shap_values = explainer.shap_values(X_test_scaled[:10])

# Feature importance for first applicant
shap_vals = shap_values[0].flatten()
feature_names = X.columns.tolist()

# Sort by absolute importance
indices = np.argsort(np.abs(shap_vals))[::-1][:8]  # top 8 features

print("Top features for applicant 1:")
for i in indices:
    direction = "↑ increases" if shap_vals[i] > 0 else "↓ decreases"
    print(f"  {feature_names[i]}: {shap_vals[i]:.4f} ({direction} default risk)")

Top features for applicant 1:
  person_home_ownership_RENT: -0.0641 (↓ decreases default risk)
  person_income: -0.0327 (↓ decreases default risk)
  loan_percent_income: -0.0245 (↓ decreases default risk)
  loan_grade_D: -0.0230 (↓ decreases default risk)
  loan_amnt: -0.0201 (↓ decreases default risk)
  loan_grade_C: -0.0150 (↓ decreases default risk)
  loan_grade_E: -0.0144 (↓ decreases default risk)
  loan_intent_VENTURE: 0.0144 (↑ increases default risk)


In [17]:
import pickle
import json

# Save model
model.save("credit_model.keras")
print("Model saved!")

# Save scaler
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
print("Scaler saved!")

# Save feature columns
feature_columns = X.columns.tolist()
with open("feature_columns.json", "w") as f:
    json.dump(feature_columns, f)
print("Feature columns saved!")

# Save a small sample of training data for SHAP background
import numpy as np
shap_background = X_train_scaled[:200]
np.save("shap_background.npy", shap_background)
print("SHAP background saved!")

Model saved!
Scaler saved!
Feature columns saved!
SHAP background saved!
